# Solved Exercises: Your First AI Agent

This notebook contains simple, beginner-friendly solutions for the exercises in the first module. We will solve 4 exercises using LangChain and Ollama.


## Setup

First, we need to load our settings and initialize the tools.


In [1]:
import os
import datetime

from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

load_dotenv()

MODEL_NAME = os.getenv("MODEL_NAME", "minimax-m2.5:cloud")
print(f"Using model: {MODEL_NAME}")


Using model: minimax-m2.5:cloud


### Exercise 1: Add a Second Tool

We will create a new tool called `get_time` and use it along with the existing `get_weather` tool.


In [2]:
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"It's always sunny in {city}!"


def get_time(city: str) -> str:
    """Get the current time for a given city."""

    now = datetime.datetime.now()
    return f"The current date and time in {city} is {now.strftime('%Y-%m-%d %H:%M:%S')}"


llm = ChatOllama(model=MODEL_NAME, temperature=0.7)
agent_with_two_tools = create_agent(
    model=llm,
    tools=[get_weather, get_time],
    system_prompt="You are a helpful assistant.",
)

response = agent_with_two_tools.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather and current time in Islamabad?",
            }
        ]
    }
)
print(response["messages"][-1].content)


The current weather in Islamabad is **sunny**! 

The current date and time in Islamabad is **March 11, 2026 at 11:15:52**.


### Exercise 2: Experiment with Temperature

We will compare how the agent behaves with different temperature settings.

- `temperature=0.0`: Very consistent and predictable.
- `temperature=1.0`: More creative and varied.


In [ ]:
def test_temperature(temp_val):
    print(f"--- Testing with Temperature: {temp_val} ---")
    llm_temp = ChatOllama(model=MODEL_NAME, temperature=temp_val)
    agent_temp = create_agent(model=llm_temp, tools=[get_weather])

    for i in range(2):
        res = agent_temp.invoke(
            {
                "messages": [
                    {"role": "user", "content": "What is the weather in Lahore?"}
                ]
            }
        )
        print(f"Run {i + 1}: {res['messages'][-1].content}\n")


test_temperature(0.0)

test_temperature(1.0)


--- Testing with Temperature: 0.0 ---
Run 1: The weather in Lahore is currently sunny! ☀️ It's described as always being sunny there. Would you like any other information about Lahore's weather?

Run 2: The weather in Lahore is currently sunny! ☀️ It's described as always being sunny there. Would you like any other information about Lahore's weather?

--- Testing with Temperature: 1.0 ---
Run 1: It looks like according to our weather information, it's always sunny in Lahore! ☀️ 

Of course, for the most accurate and current weather details, I'd recommend checking a local weather service or app for real-time updates, especially if you're planning any outdoor activities. Is there anything else you'd like to know?

Run 2: It's always sunny in Lahore! ☀️ The city is known for its generally bright and warm weather throughout the year. If you're planning to visit, be prepared for plenty of sunshine!



### Exercise 3: Break the Docstring

We will see what happens when the agent doesn't have a description (docstring) for a tool. The agent usually gets confused because it doesn't know what the tool is for!


In [ ]:
def get_weather_broken(city: str) -> str:
    # NO DOCSTRING HERE!
    return f"It's always sunny in {city}!"


try:
    agent_broken = create_agent(model=llm, tools=[get_weather_broken])

    response = agent_broken.invoke(
        {"messages": [{"role": "user", "content": "What is the weather in Lahore?"}]}
    )
    print("Response:", response["messages"][-1].content)
except Exception as e:
    print(f"Error occurred: {e}")


Error occurred: Function must have a docstring if description not provided.


### Exercise 4: Use a Model String Directly

Instead of creating a `ChatOllama` object, we can sometimes just pass the model name as a string to `create_agent` if the framework supports it.


In [ ]:
agent_string_model = create_agent(
    model=f"ollama:{MODEL_NAME}",
    tools=[get_weather],
    system_prompt="You are a simple assistant.",
)

response = agent_string_model.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Karachi?"}]}
)
print(response["messages"][-1].content)


It's always sunny in Karachi! ☀️

The weather there is typically warm and pleasant. Karachi tends to have a desert-like climate with hot summers and mild winters. Would you like more specific details about the weather or climate?
